In [ ]:
import os, joblib, json, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_curve, f1_score

MODEL_PATH = "../../models/modeloptuna.pkl"  # adjust if needed
OUT_PATH   = "../../data/predictions/predictions_repurchase.csv"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)


In [ ]:
from src.app.train.etl import UserGenerator
from src.app.train.feature_engineer import FeatureEngineer  # adapt if your class name differs

# ETL from UCI
ug = UserGenerator()
ug = ug.run_etl()
df_raw = ug.df.copy()

# Feature Engineering (match training FE)
fe = FeatureEngineer()
df_features = fe.build_all_features(df_raw)  # use your actual entrypoint

# Exact feature columns you trained on:
numeric_features = [...]       # your list from training
categorical_features = [...]   # your list from training
feat_cols = numeric_features + categorical_features

X_new = df_features[feat_cols].copy()
X_new.shape



KeyboardInterrupt



In [ ]:
pipe = joblib.load(MODEL_PATH)
proba = pipe.predict_proba(X_new)[:, 1]


In [ ]:
from src.app.train.train_mlflow_advance import TrainOptuna

# Build the same full feature table that includes the label column:
# (If your FE creates y column, set its name below)
target_column = "y_repurchase_30d"

trainer = TrainOptuna(
    df=df_features, 
    numeric_features=numeric_features, 
    categorical_features=categorical_features, 
    target_column=target_column,
    n_trials=1,  # no tuning, just for splitting
)

X_train, X_test, y_train, y_test = trainer.train_test_split_by_quantiles()

proba_test = pipe.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, proba_test)
if len(thr) > 0:
    f1s = [f1_score(y_test, (proba_test >= t).astype(int)) for t in thr]
    t_star = float(thr[int(np.argmax(f1s))])
else:
    t_star = 0.5

# Apply to current batch:
yhat = (proba >= t_star).astype(int)
t_star


In [ ]:
pred = df_features.copy()
pred["p_repurchase_30d"] = proba
pred["repurchase_flag"]  = yhat
pred.to_csv(OUT_PATH, index=False)
OUT_PATH
